# Final CERT r4.2 UEBA + DLP Pipeline

Use this notebook for the final project run on Google Colab.

Recommended runtime: `Runtime -> Change runtime type -> T4 GPU`.

Default behavior:
- Uses `/content/dlp-data` for data and generated artifacts.
- Does not download data unless you set `RUN_DOWNLOAD = True`.
- Runs clean + Isolation Forest + LSTM + DLP sensitivity + evaluation.
- `users.csv`, `ldap.csv`, and `decoy_file.csv` are optional because the r4.2 Kaggle mirror may not include them.
- Skips GA by default; run GA only after r4.2 is confirmed.
- Launches Streamlit using Colab's built-in port viewer, not ngrok.

## 1. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Taha-coder-star/DLP-PROJECt.git"
REPO_DIR = Path("/content/dlp-project")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd /content/dlp-project
!git log -1 --oneline

Cloning into '/content/dlp-project'...
remote: Enumerating objects: 10406, done.
remote: Counting objects: 100% (10406/10406), done.
remote: Compressing objects: 100% (6015/6015), done.
remote: Total 10406 (delta 4188), reused 10329 (delta 4114), pack-reused 0 (from 0)
Receiving objects: 100% (10406/10406), 20.55 MiB | 8.59 MiB/s, done.
Resolving deltas: 100% (4188/4188), done.
/content/dlp-project
e0aafec (HEAD -> main, origin/main, origin/HEAD) implementing dashboard


## 2. Configure Environment

In [ ]:
import os
import shutil
from pathlib import Path

DATA_ROOT = Path("/content/dlp-data")
REPO_DIR = Path("/content/dlp-project")

os.environ["DLP_ROOT"] = str(DATA_ROOT)
os.environ["DLP_REPO"] = str(REPO_DIR)
os.environ["PYTHONPATH"] = str(REPO_DIR)

for name in ["archive", "cleaned", "models", "plots"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage("/content")
print(f"DLP_ROOT={DATA_ROOT}")
print(f"Free disk: {free / 1024**3:.1f} GB / {total / 1024**3:.1f} GB")

DLP_ROOT=/content/dlp-data
Free disk: 87.3 GB / 107.7 GB


## 3. Install Dependencies

In [ ]:
!python -m pip install -r /content/dlp-project/requirements.txt kaggle streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.9 MB/s eta 0:00:00


## 4. Optional: Selectively Download r4.2 Raw Data

Leave `RUN_DOWNLOAD = False` if your r4.2 CSV files are already in `/content/dlp-data/archive`.

If files are missing, upload `kaggle.json` first, then set `RUN_DOWNLOAD = True`. This cell downloads the r4.2 zip, extracts only needed CSVs, then deletes the zip to save disk.

In [ ]:
RUN_DOWNLOAD = True

if RUN_DOWNLOAD:
    from google.colab import files
    import os, shutil, zipfile
    from pathlib import Path

    if not Path('/root/.kaggle/kaggle.json').exists():
        uploaded = files.upload()  # upload kaggle.json
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
        !chmod 600 /root/.kaggle/kaggle.json

    download_dir = Path('/content/r42-download')
    download_dir.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d andrihjonior/cert-insider-threat-dataset-r4-2 -p /content/r42-download

    zip_candidates = list(download_dir.glob('*.zip'))
    if not zip_candidates:
        raise FileNotFoundError('No downloaded zip found in /content/r42-download')
    zip_path = zip_candidates[0]

    out_dir = Path('/content/dlp-data/archive')
    out_dir.mkdir(parents=True, exist_ok=True)
    needed = ['email.csv', 'file.csv', 'logon.csv', 'device.csv', 'psychometric.csv', 'users.csv', 'ldap.csv', 'decoy_file.csv']

    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        for target in needed:
            matches = [n for n in names if n.endswith('/' + target) or n == target]
            if not matches:
                print('Missing in zip:', target)
                continue
            src = matches[0]
            print('Extracting:', src, '->', target)
            with z.open(src) as f_in, open(out_dir / target, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)

    zip_path.unlink()
    print('Deleted zip:', zip_path)
else:
    print('Skipping download. Using existing files in /content/dlp-data/archive.')

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/andrihjonior/cert-insider-threat-dataset-r4-2
License(s): Attribution 4.0 International (CC BY 4.0)
100% 6.65G/6.65G [01:18<00:00, 91.2MB/s]

Extracting: r4.2/email.csv -> email.csv
Extracting: r4.2/file.csv -> file.csv
Extracting: r4.2/logon.csv -> logon.csv
Extracting: r4.2/device.csv -> device.csv
Extracting: r4.2/psychometric.csv -> psychometric.csv
Missing in zip: users.csv
Missing in zip: ldap.csv
Missing in zip: decoy_file.csv
Deleted zip: /content/r42-download/cert-insider-threat-dataset-r4-2.zip


## 5. Verify Raw Files And Answer Key

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

archive = Path('/content/dlp-data/archive')
answers_src = Path('/content/dlp-project/archive/answers/answers/insiders.csv')
answers_dst = Path('/content/dlp-data/archive/answers/answers/insiders.csv')
answers_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(answers_src, answers_dst)

required = ['email.csv', 'file.csv', 'logon.csv', 'device.csv', 'psychometric.csv']
optional = ['users.csv', 'ldap.csv', 'decoy_file.csv']
missing = [name for name in required if not (archive / name).exists()]
missing_optional = [name for name in optional if not (archive / name).exists()]
print('Missing raw files:', missing or 'none')
print('Missing optional files:', missing_optional or 'none')
if missing:
    raise FileNotFoundError('Missing required raw files. Run the download cell or extract r4.2 CSVs into /content/dlp-data/archive.')

ins = pd.read_csv(answers_dst)
print('\nAnswer-key users by release:')
print(ins.groupby('dataset')['user'].nunique())

for name in required:
    p = archive / name
    print(f'{name}: {p.stat().st_size / 1024**2:.1f} MB')

Missing raw files: none
Missing optional files: ['users.csv', 'ldap.csv', 'decoy_file.csv']

Answer-key users by release:
dataset
2.0     1
3.1     2
3.2     2
4.1     3
4.2    70
5.1     4
5.2    99
6.1     5
6.2     5
Name: user, dtype: int64
email.csv: 1299.0 MB
file.csv: 184.1 MB
logon.csv: 55.8 MB
device.csv: 27.6 MB
psychometric.csv: 0.0 MB


## 6. Clear Generated Outputs For A Clean r4.2 Run

In [ ]:
CLEAR_OUTPUTS = True

if CLEAR_OUTPUTS:
    !rm -rf /content/dlp-data/cleaned /content/dlp-data/models /content/dlp-data/plots
    !mkdir -p /content/dlp-data/cleaned /content/dlp-data/models /content/dlp-data/plots
    print('Cleared generated outputs.')
else:
    print('Keeping existing generated outputs.')

Cleared generated outputs.


RUNNING THE CODE

In [ ]:
!python /content/dlp-project/colab/run_full_pipeline.py --skip-ga

Copied ground truth -> /content/dlp-data/archive/answers/answers/insiders.csv
[INFO] Optional raw files not found and will be skipped: users.csv, ldap.csv, decoy_file.csv

RUN: /usr/bin/python3 /content/dlp-project/scripts/clean_cert_email_data.py
Reading email.csv with the tolerant parser; malformed CSV rows will be skipped with a warning.
Processed chunk 1: 200000 cleaned rows
Processed chunk 2: 200000 cleaned rows
Processed chunk 3: 200000 cleaned rows
Processed chunk 4: 200000 cleaned rows
Processed chunk 5: 200000 cleaned rows
Processed chunk 6: 200000 cleaned rows
Processed chunk 7: 200000 cleaned rows
Processed chunk 8: 200000 cleaned rows
Processed chunk 9: 200000 cleaned rows
Processed chunk 10: 200000 cleaned rows
Processed chunk 11: 200000 cleaned rows
Processed chunk 12: 200000 cleaned rows
Processed chunk 13: 200000 cleaned rows
Processed chunk 14: 29979 cleaned rows
Cleaning logon data...
  Saved 854859 cleaned logon rows ->/content/dlp-data/cleaned/logon_cleaned.csv
Clea

## 8. Confirm r4.2 Was Selected

In [ ]:
import sys
sys.path.insert(0, '/content/dlp-project')
sys.path.insert(0, '/content/dlp-project/colab')

from config import CLEANED_DIR
from ground_truth import describe_selection, select_ground_truth_release

gt = select_ground_truth_release([
    CLEANED_DIR / 'email_user_daily_scored.csv',
    CLEANED_DIR / 'email_user_daily_lstm_scored.csv',
])
print(describe_selection(gt))

if gt.dataset != '4.2' or gt.match_count < 60:
    raise RuntimeError('Expected CERT 4.2 with about 70 matching insiders. Check that raw logs are r4.2, not r6.2.')
print('OK: r4.2 ground truth is aligned with the scored data.')

CERT 4.2 selected from /content/dlp-data/archive/answers/answers/insiders.csv (70/70 insider users match 1000 scored users; matches by release: 2=0, 3.1=0, 3.2=0, 4.1=0, 4.2=70, 5.1=0, 5.2=0, 6.1=0, 6.2=0)
OK: r4.2 ground truth is aligned with the scored data.


## 9. Optional: Run GA After r4.2 Is Confirmed

Only run this after Step 8 passes. Use quick mode first. Set `RUN_GA = True` when ready.

In [ ]:
RUN_GA = False
GA_QUICK = True

if RUN_GA:
    cmd = 'python /content/dlp-project/colab/ga_optimizer.py'
    if GA_QUICK:
        cmd += ' --quick'
    !$cmd
else:
    print('Skipping GA. Set RUN_GA=True after r4.2 is confirmed.')

Skipping GA. Set RUN_GA=True after r4.2 is confirmed.


## 10. Re-run Evaluation Only, No Training

In [ ]:
!python /content/dlp-project/colab/run_full_pipeline.py --skip-validate --skip-clean --skip-iforest --skip-lstm --skip-sensitivity --skip-ga

Copied ground truth -> /content/dlp-data/archive/answers/answers/insiders.csv

RUN: /usr/bin/python3 /content/dlp-project/colab/meta_model.py
Ground truth:
  CERT 4.2 selected from /content/dlp-data/archive/answers/answers/insiders.csv (70/70 insider users match 1000 scored users; matches by release: 2=0, 3.1=0, 3.2=0, 4.1=0, 4.2=70, 5.1=0, 5.2=0, 6.1=0, 6.2=0)

Feature matrix  : 1000 users × 13 signals
Insiders labelled: 70 / 1000

Training meta-model on 1000 train-split users (70 insiders)...
  5-fold CV (train users) → ROC-AUC=0.9671  AP=0.7394

  Top meta-model signal weights (LR coefficients):
    lstm_p95_norm                   +1.7519  ↑
    lstm_max_norm                   +1.6318  ↑
    max_usb_norm                    +1.4626  ↑
    file_exfil_norm                 +1.4562  ↑
    bcc_usage_norm                  -1.2548  ↓
    multi_pc_norm                   +0.9884  ↑
    usb_activity_norm               -0.8087  ↓
    flagged_day_norm                -0.4723  ↓

  Idea 13 — Meta-

## 11. Launch Dashboard In Colab

In [ ]:
import os
import subprocess
import time
from google.colab import output

env = os.environ.copy()
env['DLP_ROOT'] = '/content/dlp-data'
env['DLP_REPO'] = '/content/dlp-project'
env['PYTHONPATH'] = '/content/dlp-project'

proc = subprocess.Popen([
    'streamlit', 'run', '/content/dlp-project/app/ueba_dashboard_tabs.py',
    '--server.port', '8501',
    '--server.headless', 'true',
], env=env)

time.sleep(5)
output.serve_kernel_port_as_window(8501)
print('Keep this cell running while using the dashboard.')

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Keep this cell running while using the dashboard.
